# Fall Detection Pipeline (MoveNet + GRU) — Deployment

**Target device**: Jetson Nano | JetPack 4.6 | Python 3.6 | CUDA 10.2 | TensorRT 8.2 | PyTorch 1.9

## Project layout

```
movenet_jeston_nano/
├── movenet/                    <- everything MoveNet-related lives here
│   ├── config.py               (original author)
│   ├── train.py                (original author) train .pth on PC
│   ├── pth2onnx.py             (original author) .pth -> .onnx on PC
│   ├── lib/  scripts/  data/   (original author)
│   ├── output/                 .pth / .onnx / .engine all go here
│   ├── trt_builder.py          NEW: ONNX -> TRT engine
│   └── movenet_trt.py          NEW: TRT inference class
│
├── features.py                 fall side: 51-dim feature extractor
├── fall_model.py               fall side: GRU model + CSV dataset
├── pipeline.ipynb              <- THIS notebook (deployment)
└── train_gru.ipynb             GRU training (run on PC)
```

## End-to-end architecture

```
 video frame
      |
      v
 ┌──────────────────────┐
 │  MoveNetTRT          │   movenet/movenet_trt.py
 │  (TensorRT engine)   │
 └────────┬─────────────┘
          | (17, 3) keypoints
          v
 ┌──────────────────────┐
 │  FeatureExtractor    │   features.py
 │  norm coords + speed │   = 34 + 17 = 51 dims
 └────────┬─────────────┘
          | (51,) feature
          v
 ┌──────────────────────┐
 │  Sliding window      │   buffer (deque, maxlen=30)
 └────────┬─────────────┘
          | (30, 51)
          v
 ┌──────────────────────┐
 │  FallDetectionGRU    │   fall_model.py
 └────────┬─────────────┘
          | fall probability
          v
    alarm logic
```

## Steps in this notebook

| Step | What | Where to run |
|---|---|---|
| 0 | Environment check | Nano |
| 1 | ONNX → TRT engine | **Nano** (engine is GPU-architecture-specific) |
| 2 | MoveNet single-image test | Nano |
| 3 | End-to-end real-time inference | Nano |

**GRU training is in a separate notebook (`train_gru.ipynb`).** Train on PC, then `scp fall_gru.pth` to the Nano and run Step 3 here.

Before running heavy stuff:
```bash
sudo nvpmodel -m 0 && sudo jetson_clocks
```

## Step 0: Environment check

In [ ]:
import sys, os

# Make sure root .py modules and the movenet/ subpackage are importable.
sys.path.insert(0, os.path.abspath('.'))

print("=" * 50)
print("Environment")
print("=" * 50)
print("Python  : {}".format(sys.version.split()[0]))

for mod in ['numpy', 'cv2', 'torch']:
    try:
        m = __import__(mod)
        print("{:8s}: {}".format(mod, getattr(m, '__version__', '?')))
    except ImportError:
        print("{:8s}: NOT INSTALLED".format(mod))

try:
    import tensorrt as trt
    print("TensorRT: {}".format(trt.__version__))
except ImportError:
    print("TensorRT: NOT INSTALLED")

try:
    import pycuda.driver  # noqa
    print("PyCUDA  : OK")
except ImportError:
    print("PyCUDA  : NOT INSTALLED")

import torch
if torch.cuda.is_available():
    print("GPU     : {}".format(torch.cuda.get_device_name(0)))
else:
    print("GPU     : not available")
print("=" * 50)

---
## Step 1: ONNX → TensorRT Engine

**Must run on Nano**. `build_engine` automatically skips an existing engine (unless `rebuild=True`).

Prerequisite: run `movenet/pth2onnx.py` on the PC, then `scp` the ONNX file to the Nano.

If the Nano runs out of memory during build, enable a 4 GB swap first:
```bash
sudo fallocate -l 4G /swapfile && sudo chmod 600 /swapfile
sudo mkswap /swapfile && sudo swapon /swapfile
```

In [ ]:
from movenet.trt_builder import build_engine

# ============== Config ==============
ONNX_PATH    = "movenet/output/pose_raw.onnx"
ENGINE_PATH  = "movenet/output/pose.engine"
FP16         = True
WORKSPACE_GB = 1
REBUILD      = False         # True forces a rebuild
# ====================================

build_engine(
    onnx_path    = ONNX_PATH,
    engine_path  = ENGINE_PATH,
    fp16         = FP16,
    workspace_gb = WORKSPACE_GB,
    rebuild      = REBUILD,
    verbose      = False,
)

---
## Step 2: MoveNet single-image test

Sanity check: the engine loads, runs, and outputs a sensible (17, 3) array.
Also measures inference latency.

In [ ]:
import numpy as np
import cv2
import time

from movenet.movenet_trt import MoveNetTRT, KEYPOINT_NAMES, draw_keypoints

movenet = MoveNetTRT(ENGINE_PATH, img_size=192)

In [ ]:
# Use a real test image, fall back to a random image if none provided.
TEST_IMG = "test.jpg"     # <-- point at any image with a person

if os.path.exists(TEST_IMG):
    frame = cv2.imread(TEST_IMG)
    print("Input: {}, shape={}".format(TEST_IMG, frame.shape))
else:
    print("{} not found, using a random image to test the interface".format(TEST_IMG))
    frame = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)

# Warmup
for _ in range(3):
    movenet.infer(frame)

# Timed runs
N = 20
t0 = time.time()
for _ in range(N):
    kpts = movenet.infer(frame)
dt = (time.time() - t0) / N
print("Latency : {:.1f} ms/frame ({:.1f} FPS)".format(dt * 1000, 1.0 / dt))
print("Output  : shape={}, dtype={}".format(kpts.shape, kpts.dtype))

if os.path.exists(TEST_IMG):
    print("\nKeypoints:")
    for k, (x, y, s) in enumerate(kpts):
        print("  {:2d} {:14s} ({:6.1f}, {:6.1f}) score={:.3f}".format(
            k, KEYPOINT_NAMES[k], x, y, s))
    vis = draw_keypoints(frame, kpts)
    cv2.imwrite("test_out.jpg", vis)
    print("\nVisualization saved to: test_out.jpg")

---
## Step 3: End-to-end real-time inference

Chain MoveNet + feature extraction + GRU into a real-time loop.

Prerequisite: a trained GRU checkpoint at `models/fall_gru.pth`.
Train it on the PC using `train_gru.ipynb`, then `scp` the .pth here.

### Alarm logic
We don't fire on every frame above the threshold — that would be
noisy. Instead we accumulate a counter:
- prob > THR → counter += 1
- prob ≤ THR → counter -= 1 (clamped to 0)
- alarm fires when counter ≥ ALARM_FRAMES

This makes a single noisy frame irrelevant but a sustained 5-frame
burst trigger immediately.

In [ ]:
import torch
from collections import deque

from features import KeypointFeatureExtractor
from fall_model import FallDetectionGRU

# ============== Inference config ==============
GRU_PATH       = "models/fall_gru.pth"
VIDEO_SOURCE   = 0                 # 0 = webcam, or "test.mp4"
FALL_THRESHOLD = 0.7
ALARM_FRAMES   = 5
MAX_FRAMES     = 500               # cap to avoid hanging the notebook
SHOW_WINDOW    = True
# ==============================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gru_model, ckpt = FallDetectionGRU.load_from(GRU_PATH, map_location=device)
gru_model = gru_model.to(device).eval()
seq_len = ckpt.get('seq_len', 30)
print("GRU loaded (val_acc={:.4f}, seq_len={})".format(
    ckpt.get('val_acc', 0.0), seq_len))

extractor = KeypointFeatureExtractor()
buffer = deque(maxlen=seq_len)

In [ ]:
cap = cv2.VideoCapture(VIDEO_SOURCE)
if not cap.isOpened():
    print("[ERROR] cannot open: {}".format(VIDEO_SOURCE))
else:
    img_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    img_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print("[RUN] {}x{}, press 'q' to quit".format(img_w, img_h))

    fps_q = deque(maxlen=30)
    extractor.reset()
    buffer.clear()
    consecutive_fall = 0
    frame_idx = 0

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            t0 = time.time()

            # 1. MoveNet keypoints
            kpts = movenet.infer(frame)

            # 2. Feature extraction
            feat = extractor.extract(kpts)
            buffer.append(feat)

            # 3. GRU inference (only when buffer is full)
            fall_prob = 0.0
            if len(buffer) == seq_len:
                seq = np.stack(list(buffer), axis=0)             # (T, 51)
                seq_t = torch.from_numpy(seq).float().unsqueeze(0).to(device)
                with torch.no_grad():
                    probs = torch.softmax(gru_model(seq_t), dim=1)
                    fall_prob = float(probs[0, 1])

            # 4. Alarm logic
            if fall_prob > FALL_THRESHOLD:
                consecutive_fall += 1
            else:
                consecutive_fall = max(0, consecutive_fall - 1)
            is_alarm = consecutive_fall >= ALARM_FRAMES

            dt = time.time() - t0
            fps_q.append(dt)
            fps = 1.0 / (sum(fps_q) / len(fps_q))

            # 5. Visualization
            if SHOW_WINDOW:
                vis = draw_keypoints(frame, kpts)
                cv2.putText(vis, "FPS: {:.1f}".format(fps), (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                color = (0, 0, 255) if fall_prob > FALL_THRESHOLD else (0, 200, 0)
                cv2.putText(vis, "Fall: {:.2f}".format(fall_prob), (10, 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
                cv2.putText(vis, "Buf: {}/{}".format(len(buffer), seq_len),
                            (10, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)
                if is_alarm:
                    cv2.putText(vis, "!! FALL DETECTED !!",
                                (img_w // 2 - 180, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3)
                    cv2.rectangle(vis, (0, 0), (img_w - 1, img_h - 1), (0, 0, 255), 4)
                cv2.imshow('Fall Detection', vis)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break

            frame_idx += 1
            if MAX_FRAMES and frame_idx >= MAX_FRAMES:
                break
    finally:
        cap.release()
        if SHOW_WINDOW:
            cv2.destroyAllWindows()

    avg_fps = 1.0 / (sum(fps_q) / len(fps_q)) if fps_q else 0
    print("[DONE] {} frames, avg {:.1f} FPS".format(frame_idx, avg_fps))

---
## Production deployment (without notebook)

Once trained, the entire real-time loop can be a small standalone script:

```python
from collections import deque
import numpy as np, cv2, torch

from movenet.movenet_trt import MoveNetTRT
from features import KeypointFeatureExtractor
from fall_model import FallDetectionGRU

movenet = MoveNetTRT('movenet/output/pose.engine')
gru, ckpt = FallDetectionGRU.load_from('models/fall_gru.pth')
gru = gru.cuda().eval()
ext = KeypointFeatureExtractor()
buf = deque(maxlen=ckpt.get('seq_len', 30))

cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret: break
    kpts = movenet.infer(frame)
    buf.append(ext.extract(kpts))
    if len(buf) == buf.maxlen:
        seq = torch.from_numpy(np.stack(buf)).float().unsqueeze(0).cuda()
        with torch.no_grad():
            prob = torch.softmax(gru(seq), 1)[0, 1].item()
        if prob > 0.7:
            print('FALL!')
```

## Performance expectations (Jetson Nano, jetson_clocks)

| Stage | FPS |
|---|---|
| MoveNet TRT FP16 | 30-50 |
| + feature extraction | 30-45 |
| + GRU inference | 25-40 |

## Troubleshooting

- **Engine OOM during build**: enable 4 GB swap, close the desktop (`sudo init 3`), set workspace=1 GB
- **Low FPS**: confirm `nvpmodel -m 0 && jetson_clocks` is on
- **Keypoints off-screen**: confirm the ONNX was exported in fire717/movenet train mode (4 output heads)
- **False positives**: raise `FALL_THRESHOLD` or `ALARM_FRAMES`; add more confusing-action data (sit-down, lie-down, bend-over) to the GRU training set
- **False negatives**: more fall data from varied angles; verify Step 2 visualization first to confirm keypoints are accurate